In [124]:
import pandas as pd
import requests
import geopandas as gpd
import fiona

# Gather Crop Yield(corn,soybean,wheat) History for Tennessee from 2010 to 2024. 

In [245]:
# create private API key
import json
with open('keys.json') as fi:
    credentials = json.load(fi)
usda_crop_api_key = credentials['usda_crop_api_key']
noaa_weather_api_key = credentials['noaa_weather_api_key']


In [246]:
# Base URL for USDA NASS Quick Stats API 
crop_base_url = 'https://quickstats.nass.usda.gov/api/api_GET/'

crop_records = []

field_crops = ['CORN','SOYBEANS', 'WHEAT']

for field_crop in field_crops:
    for year in range(2010, 2025):  # Excludes 2025
        
# Define Query Parameters for filtering the data set   
# Get the keys from https://quickstats.nass.usda.gov/api from 'usage page'
        
        params = {
            'key': usda_crop_api_key,
            'commodity_desc': field_crop,
            'year': str(year),
            'state_name': 'TENNESSEE',
            'statisticcat_desc': 'YIELD', 
            'unit_desc': 'BU / ACRE'
        }
        
      # Send Get Request to USDA API with Parameters
        crop_response = requests.get(crop_base_url, params=params)
      # Convert API Response in to Python dictonary
        crop_data = crop_response.json()
        #data.keys()   # get key dictionary
        if "data" in crop_data:
            crop_records.extend(crop_data["data"])


In [247]:
print(type(all_records))
print(len(all_records))

<class 'list'>
2700


In [248]:
#Convert crop yield from list to Data frame
crop_df = pd.DataFrame(crop_records)

In [249]:
# Export Raw Crop Data frame to csv file
crop_df.to_csv('../data/Raw_Crop_df', index=False)

In [250]:
crop_df.head()

,end_code,reference_period_desc,state_alpha,county_code,class_desc,state_name,CV (%),asd_desc,zip_5,group_desc,...,region_desc,asd_code,watershed_desc,commodity_desc,unit_desc,util_practice_desc,country_name,agg_level_desc,domain_desc,state_fips_code
0,00,YEAR,TN,,ALL CLASSES,TENNESSEE,,DELTA,,FIELD CROPS,...,,10,,CORN,BU / ACRE,GRAIN,UNITED STATES,AGRICULTURAL DISTRICT,TOTAL,47
1,00,YEAR,TN,,ALL CLASSES,TENNESSEE,,WEST TENNESSEE,,FIELD CROPS,...,,20,,CORN,BU / ACRE,GRAIN,UNITED STATES,AGRICULTURAL DISTRICT,TOTAL,47
2,00,YEAR,TN,,ALL CLASSES,TENNESSEE,,WESTERN RIM,,FIELD CROPS,...,,30,,CORN,BU / ACRE,GRAIN,UNITED STATES,AGRICULTURAL DISTRICT,TOTAL,47
3,00,YEAR,TN,,ALL CLASSES,TENNESSEE,,CENTRAL BASIN,,FIELD CROPS,...,,40,,CORN,BU / ACRE,GRAIN,UNITED STATES,AGRICULTURAL DISTRICT,TOTAL,47
4,00,YEAR,TN,,ALL CLASSES,TENNESSEE,,CUMBERLAND PLATEAU,,FIELD CROPS,...,,50,,CORN,BU / ACRE,GRAIN,UNITED STATES,AGRICULTURAL DISTRICT,TOTAL,47


In [251]:
crop_df[['commodity_desc','year','state_name','statisticcat_desc','unit_desc','Value','county_name']]

,commodity_desc,year,state_name,statisticcat_desc,unit_desc,Value,county_name
0,CORN,2010,TENNESSEE,YIELD,BU / ACRE,124.4,
1,CORN,2010,TENNESSEE,YIELD,BU / ACRE,116.7,
2,CORN,2010,TENNESSEE,YIELD,BU / ACRE,101.1,
3,CORN,2010,TENNESSEE,YIELD,BU / ACRE,115.3,
4,CORN,2010,TENNESSEE,YIELD,BU / ACRE,129.7,
...,...,...,...,...,...,...,...
2695,WHEAT,2024,TENNESSEE,YIELD,BU / ACRE,75,
2696,WHEAT,2024,TENNESSEE,YIELD,BU / ACRE,76,
2697,WHEAT,2024,TENNESSEE,YIELD,BU / ACRE,76,
2698,WHEAT,2024,TENNESSEE,YIELD,BU / ACRE,76,


In [252]:
crop_df["county_name"].unique()

array(['', 'DYER', 'LAKE', 'LAUDERDALE', 'OBION', 'SHELBY', 'TIPTON',
       'BENTON', 'CARROLL', 'CHESTER', 'CROCKETT', 'DECATUR', 'FAYETTE',
       'GIBSON', 'HARDEMAN', 'HARDIN', 'HAYWOOD', 'HENDERSON', 'HENRY',
       'MCNAIRY', 'MADISON', 'WEAKLEY', 'OTHER (COMBINED) COUNTIES',
       'CHEATHAM', 'HICKMAN', 'HUMPHREYS', 'LAWRENCE', 'MONTGOMERY',
       'PERRY', 'ROBERTSON', 'STEWART', 'WAYNE', 'BEDFORD', 'CANNON',
       'CLAY', 'DE KALB', 'GILES', 'LINCOLN', 'MACON', 'MARSHALL',
       'MAURY', 'RUTHERFORD', 'SUMNER', 'WILLIAMSON', 'WILSON', 'COFFEE',
       'CUMBERLAND', 'FRANKLIN', 'GRUNDY', 'MARION', 'OVERTON',
       'SEQUATCHIE', 'VAN BUREN', 'WARREN', 'WHITE', 'BLOUNT', 'BRADLEY',
       'CLAIBORNE', 'COCKE', 'GRAINGER', 'GREENE', 'HAMBLEN', 'HAWKINS',
       'JEFFERSON', 'MCMINN', 'MEIGS', 'MONROE', 'POLK', 'SEVIER',
       'DICKSON', 'JACKSON', 'SMITH', 'BLEDSOE', 'FENTRESS', 'PICKETT',
       'HAMILTON', 'SULLIVAN', 'CARTER', 'JOHNSON', 'RHEA', 'WASHINGTON',
       'TROU

In [253]:
(crop_df["county_name"] == "").sum()

np.int64(475)

In [254]:
crop_df[
    ~crop_df["county_name"].isin(
        ["OTHER COUNTIES", "OTHER (COMBINED) COUNTIES"]
    )
]

,end_code,reference_period_desc,state_alpha,county_code,class_desc,state_name,CV (%),asd_desc,zip_5,group_desc,...,region_desc,asd_code,watershed_desc,commodity_desc,unit_desc,util_practice_desc,country_name,agg_level_desc,domain_desc,state_fips_code
0,00,YEAR,TN,,ALL CLASSES,TENNESSEE,,DELTA,,FIELD CROPS,...,,10,,CORN,BU / ACRE,GRAIN,UNITED STATES,AGRICULTURAL DISTRICT,TOTAL,47
1,00,YEAR,TN,,ALL CLASSES,TENNESSEE,,WEST TENNESSEE,,FIELD CROPS,...,,20,,CORN,BU / ACRE,GRAIN,UNITED STATES,AGRICULTURAL DISTRICT,TOTAL,47
2,00,YEAR,TN,,ALL CLASSES,TENNESSEE,,WESTERN RIM,,FIELD CROPS,...,,30,,CORN,BU / ACRE,GRAIN,UNITED STATES,AGRICULTURAL DISTRICT,TOTAL,47
3,00,YEAR,TN,,ALL CLASSES,TENNESSEE,,CENTRAL BASIN,,FIELD CROPS,...,,40,,CORN,BU / ACRE,GRAIN,UNITED STATES,AGRICULTURAL DISTRICT,TOTAL,47
4,00,YEAR,TN,,ALL CLASSES,TENNESSEE,,CUMBERLAND PLATEAU,,FIELD CROPS,...,,50,,CORN,BU / ACRE,GRAIN,UNITED STATES,AGRICULTURAL DISTRICT,TOTAL,47
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2695,00,YEAR,TN,,WINTER,TENNESSEE,,,,FIELD CROPS,...,,,,WHEAT,BU / ACRE,ALL UTILIZATION PRACTICES,UNITED STATES,STATE,TOTAL,47
2696,00,YEAR - AUG FORECAST,TN,,WINTER,TENNESSEE,,,,FIELD CROPS,...,,,,WHEAT,BU / ACRE,ALL UTILIZATION PRACTICES,UNITED STATES,STATE,TOTAL,47
2697,00,YEAR - JUL FORECAST,TN,,WINTER,TENNESSEE,,,,FIELD CROPS,...,,,,WHEAT,BU / ACRE,ALL UTILIZATION PRACTICES,UNITED STATES,STATE,TOTAL,47
2698,00,YEAR - JUN FORECAST,TN,,WINTER,TENNESSEE,,,,FIELD CROPS,...,,,,WHEAT,BU / ACRE,ALL UTILIZATION PRACTICES,UNITED STATES,STATE,TOTAL,47


# Later Clean Crops data frame 

# Gather weather History from 2010 to 2024

Get Api from this link https://www.ncei.noaa.gov/cdo-web/webservices/v2#data for the weather history

In [255]:
# Call Stations end point to get stationsid to find the Countiesafter merging it weather data
# stations_base_url = "https://www.ncei.noaa.gov/cdo-web/api/v2/stations"

# headers = {
#     "token": noaa_weather_api_key
# }

# params = {
#     "datasetid": "GSOM",
#     "locationid": "FIPS:47",
#     "startdate": "2010-01-01",
#     "enddate": "2024-12-31",
#     "limit": 1000
# }

# req = requests.get(stations_base_url,headers=headers,params=params)
# stations = req.json()
# stations.keys()
# stations_repo = stations['results']

In [256]:

#stations = pd.DataFrame(stations_repo)


In [257]:
# Call Stations end point to get stationsid to find the Countiesafter merging it weather data
stations_base_url = "https://www.ncei.noaa.gov/cdo-web/api/v2/stations"

# Authentication token
headers = {
    "token": noaa_weather_api_key
}

# Query Parameters to filter stations
params = {
    "datasetid": "GSOM",         # Monthly Summary Data
    "locationid": "FIPS:47",     # TN state Fips code 47
    "startdate": "2010-01-01",   # Start date filter for station availabilty
    "enddate": "2024-12-31",     # End date filter
    "limit": 1000,               # Max Records per one API Request
    "offset": 1                  # pagination start index
}

# List to store all station records
all_stations = []

# Loop to handle pagination
while True:

    # Make API request to NOAA stations endpoint
    response = requests.get(stations_base_url, headers=headers, params=params)

    # Stop if API call fails
    if response.status_code != 200:
        print("Request failed:", response.text)
        break

    # Convert JSON response to Python dictionary
    data = response.json()

    # Extract station records from response
    results = data.get("results", [])

    # stop if no more data
    if not results:
        break

    # Add current page results to all_stations list
    all_stations.extend(results)

    # if last page less than limit, stop
    if len(results) < params["limit"]:
        break

    #Move to next page using offset pagination
    params["offset"] += params["limit"]

# Convert all stations list to Dataframe
stations_df = pd.DataFrame(all_stations)

In [258]:
stations_df.head()

,elevation,mindate,maxdate,latitude,name,datacoverage,id,elevationUnit,longitude
0,282.5,2008-01-01,2015-06-01,35.011389,"ARDMORE 1.3 NE, TN US",0.3224,GHCND:US1CHARM185,METERS,-86.839167
1,270.1,2007-05-01,2026-05-01,36.019079,"OAK RIDGE 5.7 NE, TN US",0.8864,GHCND:US1TNAN0003,METERS,-84.221136
2,353.0,2007-10-01,2026-05-01,36.199700,"NORRIS 0.6 NW, TN US",0.9063,GHCND:US1TNAN0008,METERS,-84.076500
3,260.3,2009-08-01,2021-08-01,36.021157,"OAK RIDGE 5.7 NE, TN US",0.9102,GHCND:US1TNAN0009,METERS,-84.223414
4,249.0,2012-11-01,2026-05-01,36.019905,"CLINTON 4.9 S, TN US",0.4357,GHCND:US1TNAN0012,METERS,-84.125534


In [259]:
stations_df.shape

(1165, 9)

In [260]:
# Export Raw stations Data frame to csv file
stations_df.to_csv('../data/Raw_stations_df', index=False)

spatialjoins, find the nearest station for there are no counties

Get Api from this link https://www.ncei.noaa.gov/cdo-web/webservices/v2#data for the weather history

In [261]:
# weather_base_url = "https://www.ncei.noaa.gov/cdo-web/api/v2/data"

# headers = {
#     "token": noaa_weather_api_key
# }

# params = {
                
#             "datasetid": "GSOM",
#             "locationid": "FIPS:47",
#             "startdate": "2010-01-01",
#             "enddate": "2017-12-31",
#             "datatypeid": "TMIN",
#             "limit": 1000
#         }
# weather_response = requests.get(weather_base_url,headers=headers, params=params)
# weather = weather_response.json()
# weather.keys()
# weather_response

In [262]:
#weather_response.text

In [263]:
# weather_repo = weather['results']
# pd.DataFrame(weather_repo)

In [264]:
# Base URL for NCEI NOAA WEATHER API 

weather_base_url = 'https://www.ncei.noaa.gov/cdo-web/api/v2/data'

headers = {
    "token": noaa_weather_api_key
}

all_weather = []

dataTypes = ['PRCP','TMAX', 'TMIN']

for dataType in dataTypes:
    
    for year in range(2011, 2025):
        offset = 1  # pagination start
        
        while True:
        # Define Query Parameters for filtering the data set  
        # Get the keys from https://www.ncei.noaa.gov/cdo-web/webservices/v2#data 
            params = {
                "datasetid": "GSOM",
                "locationid": "FIPS:47",
                "startdate": f"{year}-01-01",
                "enddate": f"{year}-12-31",
                "datatypeid": dataType,
                "units" : "standard",
                "limit": 1000,
                "offset": offset
            }
                
              # Send Get Request to ncei NOAA  API with Parameters
            weather_response = requests.get(weather_base_url, headers=headers, params=params, timeout=60)
              # Convert API Response in to Python dictonary
            weather_data = weather_response.json()
            if weather_response.status_code != 200:
                print(f"Failed: {dataType} {year}")
                break

            results = weather_data.get("results",[])

            if len(results) == 0:
                break

            all_weather.extend(results)

            if len(results) < 1000:
                break

            offset += 1000

            #weather_data.keys()
            #if "results" in weather_data:
                #all_weather.extend(weather_data["results"])
            #offset += 1000
weather_df = pd.DataFrame(all_weather)

In [266]:
#Export Raw weather Data frame to csv file
weather_df.to_csv('../data/Raw_weather_df', index=False)

In [267]:
weather_df.head()

,date,datatype,station,attributes,value
0,2011-01-01T00:00:00,PRCP,GHCND:US1TNAN0003,",,,N",4.19
1,2011-01-01T00:00:00,PRCP,GHCND:US1TNAN0008,",,,N",3.89
2,2011-01-01T00:00:00,PRCP,GHCND:US1TNAN0009,",,,N",4.10
3,2011-01-01T00:00:00,PRCP,GHCND:US1TNBF0003,"5,,,N",4.36
4,2011-01-01T00:00:00,PRCP,GHCND:US1TNBF0004,",,,N",4.34


In [268]:
weather_df.shape

(114517, 5)

In [219]:
counties = gpd.read_file('../data/tl_2023_us_county.zip')

In [220]:
tn_counties = counties[counties["STATEFP"] == "47"]

In [221]:
tn_counties

,STATEFP,COUNTYFP,COUNTYNS,GEOID,GEOIDFQ,NAME,NAMELSAD,LSAD,CLASSFP,MTFCC,CSAFP,CBSAFP,METDIVFP,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry
29,47,065,01639749,47065,0500000US47065,Hamilton,Hamilton County,06,H1,G4020,174,16860,None,A,1404197535,86784458,+35.1634720,-085.2018432,"POLYGON ((-85.17523 34.98598, -85.17663 34.986..."
64,47,115,01639770,47115,0500000US47115,Marion,Marion County,06,H1,G4020,174,16860,None,A,1290472798,36484703,+35.1334215,-085.6183990,"POLYGON ((-85.86396 34.99316, -85.86395 34.993..."
67,47,185,01639800,47185,0500000US47185,White,White County,06,H1,G4020,None,18260,None,A,975592109,7113368,+35.9270486,-085.4557854,"MULTIPOLYGON (((-85.6446 36.01505, -85.64392 3..."
127,47,129,01639778,47129,0500000US47129,Morgan,Morgan County,06,H1,G4020,315,28940,None,A,1352439680,823018,+36.1386970,-084.6392616,"POLYGON ((-84.45834 36.18654, -84.45834 36.186..."
180,47,013,01639728,47013,0500000US47013,Campbell,Campbell County,06,H1,G4020,315,28940,None,A,1243685306,46425147,+36.4015922,-084.1592495,"POLYGON ((-84.09196 36.22261, -84.09202 36.222..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3080,47,039,01639739,47039,0500000US47039,Decatur,Decatur County,06,H1,G4020,None,None,None,A,864741776,28501002,+35.6031362,-088.1102917,"POLYGON ((-87.97643 35.50673, -87.9767 35.5058..."
3085,47,119,01639772,47119,0500000US47119,Maury,Maury County,06,H1,G4020,400,34980,None,A,1587995804,6344537,+35.6156963,-087.0777632,"POLYGON ((-86.82054 35.63207, -86.8207 35.6308..."
3103,47,047,01639742,47047,0500000US47047,Fayette,Fayette County,06,H1,G4020,368,32820,None,A,1823539295,3696802,+35.1969933,-089.4138027,"POLYGON ((-89.3388 35.40016, -89.33509 35.4000..."
3108,47,063,01648576,47063,0500000US47063,Hamblen,Hamblen County,06,H1,G4020,315,34100,None,A,417476191,37852326,+36.2183967,-083.2660711,"POLYGON ((-83.24943 36.28716, -83.24935 36.287..."


In [222]:
tn_counties = tn_counties[["NAME", "geometry"]]
tn_counties = tn_counties.rename(columns={"NAME": "county"})

In [223]:
tn_counties

,county,geometry
29,Hamilton,"POLYGON ((-85.17523 34.98598, -85.17663 34.986..."
64,Marion,"POLYGON ((-85.86396 34.99316, -85.86395 34.993..."
67,White,"MULTIPOLYGON (((-85.6446 36.01505, -85.64392 3..."
127,Morgan,"POLYGON ((-84.45834 36.18654, -84.45834 36.186..."
180,Campbell,"POLYGON ((-84.09196 36.22261, -84.09202 36.222..."
...,...,...
3080,Decatur,"POLYGON ((-87.97643 35.50673, -87.9767 35.5058..."
3085,Maury,"POLYGON ((-86.82054 35.63207, -86.8207 35.6308..."
3103,Fayette,"POLYGON ((-89.3388 35.40016, -89.33509 35.4000..."
3108,Hamblen,"POLYGON ((-83.24943 36.28716, -83.24935 36.287..."


In [224]:
stations_gdf = gpd.GeoDataFrame(
    stations_df,
    geometry=gpd.points_from_xy(stations_df.longitude, stations_df.latitude),
    crs="EPSG:4326"
)

In [225]:
stations_gdf = stations_gdf.to_crs(tn_counties.crs)

In [226]:
stations_with_county = gpd.sjoin(
    stations_gdf,
    tn_counties,
    how="left",
    predicate="within"
)

In [202]:
weather_df["date"] = pd.to_datetime(weather_df["date"])
weather_df["year"] = weather_df["date"].dt.year

In [203]:
weather_df

,date,datatype,station,attributes,value,year
0,2011-01-01,PRCP,GHCND:US1TNAN0003,",,,N",4.19,2011
1,2011-01-01,PRCP,GHCND:US1TNAN0008,",,,N",3.89,2011
2,2011-01-01,PRCP,GHCND:US1TNAN0009,",,,N",4.10,2011
3,2011-01-01,PRCP,GHCND:US1TNBF0003,"5,,,N",4.36,2011
4,2011-01-01,PRCP,GHCND:US1TNBF0004,",,,N",4.34,2011
...,...,...,...,...,...,...
114512,2024-12-01,TMIN,GHCND:USW00013891,",,,W",33.90,2024
114513,2024-12-01,TMIN,GHCND:USW00013893,",,,W",40.00,2024
114514,2024-12-01,TMIN,GHCND:USW00013897,",,,W",36.80,2024
114515,2024-12-01,TMIN,GHCND:USW00053868,",,,W",34.20,2024


In [204]:
weather_pivot = weather_df.pivot_table(
    index=["station", "year"],
    columns="datatype",
    values="value",
    aggfunc="mean"
).reset_index()

In [205]:
weather_pivot

datatype,station,year,PRCP,TMAX,TMIN
0,GHCND:US1CHARM185,2014,4.870000,NaN,NaN
1,GHCND:US1CHARM185,2015,4.551667,NaN,NaN
2,GHCND:US1TNAN0003,2011,5.853000,NaN,NaN
3,GHCND:US1TNAN0003,2012,4.128182,NaN,NaN
4,GHCND:US1TNAN0003,2013,5.736667,NaN,NaN
...,...,...,...,...,...
7305,GHCND:USW00063855,2020,5.874167,65.183333,45.691667
7306,GHCND:USW00063855,2021,4.937500,65.033333,45.066667
7307,GHCND:USW00063855,2022,5.442500,64.933333,43.875000
7308,GHCND:USW00063855,2023,4.472500,66.008333,45.191667


In [227]:
weather_station = weather_pivot.merge(
    stations_with_county[["id", "county"]],
    left_on="station",
    right_on="id",
    how="left"
)

In [229]:
weather_station

,station,year,PRCP,TMAX,TMIN,id,county
0,GHCND:US1CHARM185,2014,4.870000,NaN,NaN,GHCND:US1CHARM185,Giles
1,GHCND:US1CHARM185,2015,4.551667,NaN,NaN,GHCND:US1CHARM185,Giles
2,GHCND:US1TNAN0003,2011,5.853000,NaN,NaN,GHCND:US1TNAN0003,Anderson
3,GHCND:US1TNAN0003,2012,4.128182,NaN,NaN,GHCND:US1TNAN0003,Anderson
4,GHCND:US1TNAN0003,2013,5.736667,NaN,NaN,GHCND:US1TNAN0003,Anderson
...,...,...,...,...,...,...,...
7305,GHCND:USW00063855,2020,5.874167,65.183333,45.691667,GHCND:USW00063855,Cumberland
7306,GHCND:USW00063855,2021,4.937500,65.033333,45.066667,GHCND:USW00063855,Cumberland
7307,GHCND:USW00063855,2022,5.442500,64.933333,43.875000,GHCND:USW00063855,Cumberland
7308,GHCND:USW00063855,2023,4.472500,66.008333,45.191667,GHCND:USW00063855,Cumberland


In [230]:
county_weather = weather_station.groupby(
    ["county", "year"]
).agg({
    "PRCP": "mean",
    "TMAX": "mean",
    "TMIN": "mean"
}).reset_index()

In [231]:
county_weather

,county,year,PRCP,TMAX,TMIN
0,Anderson,2011,6.205546,70.177778,47.738889
1,Anderson,2012,3.781006,71.775000,48.894444
2,Anderson,2013,5.645399,67.644444,47.133333
3,Anderson,2014,4.382334,68.792424,46.710606
4,Anderson,2015,4.803946,70.230556,48.041414
...,...,...,...,...,...
1290,Wilson,2020,4.977994,72.804356,50.714015
1291,Wilson,2021,5.067313,66.310000,44.554167
1292,Wilson,2022,4.578092,70.199242,45.154545
1293,Wilson,2023,3.643431,72.842778,48.564545


# Gather Soil Data

In [178]:
soil_keys = gpd.read_file("../data/gSSURGO_TN.gdb", layer="MUPOLYGON") #MULTIPOLYGON - a single geometric object that represents a collection of multiple, non-overlapping polygons treated as one feature

In [181]:
soil_keys.head()

,AREASYMBOL,SPATIALVER,MUSYM,MUKEY,Shape_Length,Shape_Area,geometry
0,TN091,5.0,CaF,526516,5236.926602,534416.62,"MULTIPOLYGON (((1259534.4 1600055.8, 1259510.8..."
1,TN091,5.0,CcE,526518,2847.382752,223007.82,"MULTIPOLYGON (((1244721.2 1577069.9, 1244679.2..."
2,TN091,5.0,DtE,526529,2299.663378,79289.81,"MULTIPOLYGON (((1259721.8 1591012.2, 1259693 1..."
3,TN091,5.0,KeC,526541,1132.074101,77883.95,"MULTIPOLYGON (((1258520.1 1590386.1, 1258526.7..."
4,TN091,5.0,CaE,526515,964.949459,48297.57,"MULTIPOLYGON (((1242604.1 1578393.6, 1242606.3..."


In [179]:
muaggatt_df =gpd.read_file("../data/gSSURGO_TN.gdb", layer="Muaggatt")

In [182]:
muaggatt_df.head()

,musym,muname,mustatus,slopegraddcp,slopegradwta,brockdepmin,wtdepannmin,wtdepaprjunmin,flodfreqdcd,flodfreqmax,...,engsldcp,englrsdcd,engcmssdcd,engcmssmp,urbrecptdcd,urbrecptwta,forpehrtdcp,hydclprs,awmmfpwwta,mukey
0,AcF,"Ashe-Cleveland-Rock outcrop complex, 50 to 95 ...",None,73.0,73.0,0.0,NaN,NaN,None,None,...,Very limited,Very limited,Fair,Not rated,Very limited,1.00,Erosion hazard very severe,0,1.000,526504
1,AsE,"Ashe gravelly fine sandy loam, 12 to 25 percen...",None,19.0,19.0,81.0,NaN,NaN,None,None,...,Very limited,Very limited,Fair,Fair,Somewhat limited,0.32,Erosion hazard moderate,0,1.000,526505
2,AsF,"Ashe gravelly fine sandy loam, 25 to 65 percen...",None,45.0,45.0,81.0,NaN,NaN,None,None,...,Very limited,Very limited,Fair,Fair,Very limited,1.00,Erosion hazard severe,0,1.000,526506
3,BeC,"Bledsoe silt loam, 5 to 12 percent slopes",None,9.0,9.0,NaN,NaN,NaN,None,None,...,Very limited,Very limited,Poor,Poor,Very limited,1.00,Erosion hazard severe,0,0.412,526507
4,BeD,"Bledsoe silt loam, 12 to 20 percent slopes",None,16.0,16.0,NaN,NaN,NaN,None,None,...,Very limited,Very limited,Poor,Poor,Very limited,1.00,Erosion hazard very severe,0,1.000,526508


In [180]:
mugdp_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6870 entries, 0 to 6869
Data columns (total 40 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   musym           6870 non-null   object 
 1   muname          6870 non-null   object 
 2   mustatus        0 non-null      object 
 3   slopegraddcp    6686 non-null   float32
 4   slopegradwta    6697 non-null   float32
 5   brockdepmin     2021 non-null   float64
 6   wtdepannmin     1890 non-null   float64
 7   wtdepaprjunmin  1322 non-null   float64
 8   flodfreqdcd     6608 non-null   object 
 9   flodfreqmax     6638 non-null   object 
 10  pondfreqprs     6870 non-null   object 
 11  aws025wta       6564 non-null   float32
 12  aws050wta       6564 non-null   float32
 13  aws0100wta      6564 non-null   float32
 14  aws0150wta      6564 non-null   float32
 15  drclassdcd      6487 non-null   object 
 16  drclasswettest  6567 non-null   object 
 17  hydgrpdcd       6490 non-null   o

In [152]:
mugdp_df['muname']

0       Ashe-Cleveland-Rock outcrop complex, 50 to 95 ...
1       Ashe gravelly fine sandy loam, 12 to 25 percen...
2       Ashe gravelly fine sandy loam, 25 to 65 percen...
3               Bledsoe silt loam, 5 to 12 percent slopes
4              Bledsoe silt loam, 12 to 20 percent slopes
                              ...                        
6865                                                Water
6866    Waverly silt loam, 0 to 2 percent slopes, occa...
6867                                Mines and Gravel Pits
6868    Adler silt loam, 0 to 2 percent slopes, occasi...
6869    Roellen silty clay loam, 0 to 1 percent slopes...
Name: muname, Length: 6870, dtype: object

In [183]:
soil_gdf = soil_keys.merge(muaggatt_df,left_on="MUKEY",right_on="mukey",how="left")

In [184]:
soil_gdf

,AREASYMBOL,SPATIALVER,MUSYM,MUKEY,Shape_Length,Shape_Area,geometry,musym,muname,mustatus,...,engsldcp,englrsdcd,engcmssdcd,engcmssmp,urbrecptdcd,urbrecptwta,forpehrtdcp,hydclprs,awmmfpwwta,mukey
0,TN091,5.0,CaF,526516,5236.926602,534416.620,"MULTIPOLYGON (((1259534.4 1600055.8, 1259510.8...",CaF,"Calvin channery silt loam, 35 to 50 percent sl...",None,...,Very limited,Very limited,Poor,Poor,Very limited,1.000,Erosion hazard severe,0,1.000,526516
1,TN091,5.0,CcE,526518,2847.382752,223007.820,"MULTIPOLYGON (((1244721.2 1577069.9, 1244679.2...",CcE,"Cataska channery silt loam, 20 to 35 percent s...",None,...,Very limited,Very limited,Poor,Fair,Very limited,1.000,Erosion hazard moderate,0,1.000,526518
2,TN091,5.0,DtE,526529,2299.663378,79289.810,"MULTIPOLYGON (((1259721.8 1591012.2, 1259693 1...",DtE,"Ditney sandy loam, 20 to 35 percent slopes",None,...,Very limited,Very limited,Fair,Fair,Very limited,1.000,Erosion hazard moderate,0,1.000,526529
3,TN091,5.0,KeC,526541,1132.074101,77883.950,"MULTIPOLYGON (((1258520.1 1590386.1, 1258526.7...",KeC,"Keener loam, 5 to 12 percent slopes",None,...,Very limited,Very limited,Poor,Poor,Very limited,1.000,Erosion hazard severe,0,0.622,526541
4,TN091,5.0,CaE,526515,964.949459,48297.570,"MULTIPOLYGON (((1242604.1 1578393.6, 1242606.3...",CaE,"Calvin channery silt loam, 20 to 35 percent sl...",None,...,Very limited,Very limited,Poor,Poor,Very limited,1.000,Erosion hazard moderate,0,1.000,526515
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1314995,TN157,11.0,Ad,3116427,1840.030629,52411.240,"MULTIPOLYGON (((562158.7 1388289.8, 562161.1 1...",Ad,"Adler silt loam, 0 to 2 percent slopes, occasi...",None,...,Very limited,Very limited,Poor,Poor,Somewhat limited,0.062,Erosion hazard slight,0,0.999,3116427
1314996,TN157,11.0,W,567324,104.434694,770.015,"MULTIPOLYGON (((541549 1387737.4, 541530.9 138...",W,Water,None,...,Not rated,Not rated,Not rated,Not rated,Not rated,NaN,Not rated,0,NaN,567324
1314997,TN157,11.0,GaB2,567294,1841.788500,53532.270,"MULTIPOLYGON (((551479.7 1387915.7, 551483.1 1...",GaB2,"Grenada silt loam, 2 to 5 percent slopes, mode...",None,...,Somewhat limited,Somewhat limited,Poor,Poor,Somewhat limited,0.082,Erosion hazard moderate,0,1.000,567294
1314998,TN157,11.0,Ad,3116427,13177.131883,554053.995,"MULTIPOLYGON (((552534.8 1388238.6, 552519.3 1...",Ad,"Adler silt loam, 0 to 2 percent slopes, occasi...",None,...,Very limited,Very limited,Poor,Poor,Somewhat limited,0.062,Erosion hazard slight,0,0.999,3116427


In [185]:
soil_gdf.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 1315000 entries, 0 to 1314999
Data columns (total 47 columns):
 #   Column          Non-Null Count    Dtype   
---  ------          --------------    -----   
 0   AREASYMBOL      1315000 non-null  object  
 1   SPATIALVER      1315000 non-null  float64 
 2   MUSYM           1315000 non-null  object  
 3   MUKEY           1315000 non-null  object  
 4   Shape_Length    1315000 non-null  float64 
 5   Shape_Area      1315000 non-null  float64 
 6   geometry        1315000 non-null  geometry
 7   musym           1315000 non-null  object  
 8   muname          1315000 non-null  object  
 9   mustatus        0 non-null        object  
 10  slopegraddcp    1273270 non-null  float32 
 11  slopegradwta    1274060 non-null  float32 
 12  brockdepmin     362838 non-null   float64 
 13  wtdepannmin     373441 non-null   float64 
 14  wtdepaprjunmin  253357 non-null   float64 
 15  flodfreqdcd     1267934 non-null  object  
 16  flodfreqma

In [186]:
soil_gdf['aws025wta']

0          3.20
1          2.96
2          3.01
3          3.79
4          3.20
           ... 
1314995    5.62
1314996     NaN
1314997    5.54
1314998    5.62
1314999    5.50
Name: aws025wta, Length: 1315000, dtype: float32

In [232]:
soil_gdf = soil_gdf.to_crs(tn_counties.crs)

In [ ]:
soil_with_county = gpd.sjoin(
    soil_gdf,
    tn_counties,
    how="left",
    predicate="intersects"
)